<a href="https://colab.research.google.com/github/rishabh135/2015/blob/master/Visualizing_token_embeddings_in_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing the Token Embedding Space of an LLM

> **Disclaimer**: *unlike other posts in this blog that actually served some purpose, this is just a random idea I had and am implementing for fun. So if your question is "why should I want to visualize the vocabulary of an LLM?", I don't have an answer* 😄

---

I recently realized I have never visualized the embedding space of a large language model (LLM). And while it's perfectly reasonable to go through life without ever seeing the inside of an LLM's vocabulary, I personally find it satisfying to make abstract things more concrete — especially when they're hiding in 4096-dimensional space.

LLMs are trained on vast corpora of text. To process text, they first tokenize it — splitting it into units (subwords, words, or even characters) — and then represent each token as a dense vector in a high-dimensional space. These vectors are parameters of the model, learned during training.

The size of the vocabulary (i.e., how many distinct tokens the model knows) is a hyperparameter. For example, LLaMA 2: \~32,000 tokens, LLaMA 3: \~128,000 tokens, LLaMA 4: up to 200,000 tokens and so on. The token embedding dimension, that is the number of elements of each vector representing a token, (e.g., 4096) is another hyperparameter.

So if we take a LLaMA 3 model with an embedding size of 4096 and a vocabulary of 128k tokens, we’re dealing with 128k points in a  4096-dimensional space — not exactly human-interpretable.

But what if we reduce the dimensionality? We’ll lose information, sure, but it might still give us interesting insights.

---

### 🔧 What We'll Do

* Use PCA to project the 4096-dimensional token vectors into 3D space (so we can plot them).
* Use a small corpus (a slice of Wikipedia) to filter only the tokens that appear in it — otherwise, we’d be visualizing 128k points.
* Plot token embeddings interactively with Plotly.
* (Extras below!)





In [ ]:
# Let's first install some dependecies for later
!pip install datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==1

In [ ]:
# If you want to use some gated model from hf like Llama, you'll need an access token
# Else you can use Phi 1.5 that does not require this
hf_token = "<your token here! not needed if you use phi-1.5 :)>"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model and tokenizer
model_name = "meta-llama/Llama-3.2-1B"  # Change to your preferred model, for some model's you'll need a HF token
# model_name = "meta-llama/Llama-2-7b-hf"
# model_name = "microsoft/phi-1.5"

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.bfloat16, token=hf_token).eval()


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

To avoid plotting all 128k tokens, we’ll just extract those that actually appear in some real text. (If you want to visualize the **entire** vocabulary, I'll leave the code at the end of the post, but consider that might generate a rather large plot.)

Let's download a small corpus and extract the tokens that appear there. We’ll use a fraction of the Wikitext dataset. It’s mostly English with Latin characters, which makes token visualization more intuitive.

In [ ]:
from datasets import load_dataset

# Load a small corpus, 50% of the training dataset of wikitext
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:30%]", token=hf_token)
corpus = " ".join(dataset["text"])

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

We tokenize the corpus (text -> list of tokens) and count how often each token appears. This helps us later if we want to color or size points by frequency.

In [ ]:
from collections import Counter

# Tokenize the corpus
input_ids = tokenizer(corpus, return_tensors="pt")["input_ids"].flatten()
unique_ids = sorted(set(input_ids.tolist()))
freqs = Counter(input_ids.tolist())


Token indices sequence length is longer than the specified maximum sequence length for this model (730112 > 131072). Running this sequence through the model will result in indexing errors


Each token ID corresponds to a row in the model’s embedding matrix (`model.embed_tokens`). These vectors are what we want to project and visualize.

In [ ]:
# Extract embeddings only for tokens in corpus
embedding_weights = model.model.embed_tokens(torch.tensor(unique_ids).unsqueeze(0).to(model.device)).squeeze(0)

We project the high-dimensional vectors (e.g., 4096D) into 3D using [PCA](). This means that each token embedding will go from 4096 -> 3 dimensions.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Reduce to 3D
reduce_dims = PCA(n_components=3)
# reduce_dims = TSNE(n_components=3)

embeddings_3d = reduce_dims.fit_transform(embedding_weights.detach().cpu().float())

# We convert token IDs back into human-readable text and collect their frequencies.
tokens = [tokenizer.decode([i]) for i in unique_ids]
sizes = [freqs[i] for i in unique_ids]



Now let’s put it all together in an interactive 3D plot. I'm clipping the tokens that appear more than 100 times because they are outliers that would cause all the other points to have the same color.

In [ ]:
import plotly.io as pio
import plotly.express as px

pio.renderers.default = 'colab'

# Plot with Plotly
fig = px.scatter_3d(
    x=embeddings_3d[:, 0],
    y=embeddings_3d[:, 1],
    z=embeddings_3d[:, 2],
    hover_name=tokens,
    color={k: v if v < 100 else 100 for k, v in freqs.items()}, # cutting the frequency to 100 because there are too many outliers
    #size={k: v if v < 100 else 100 for k, v in freqs.items()},
    title="Token Embeddings (Filtered by Corpus)",
)

fig.update_traces(marker=dict(size=2))
fig.update_coloraxes(showscale=False)

fig.update_layout(
    scene=dict(
        aspectmode="cube",
        xaxis=dict(backgroundcolor="white", gridcolor="lightgray"),
        yaxis=dict(backgroundcolor="white", gridcolor="lightgray"),
        zaxis=dict(backgroundcolor="white", gridcolor="lightgray"),
    ),
    paper_bgcolor="#ebebeb",  # around the plot
    font_color="white"      # text, labels, hover
)

fig.show()

## Extra 1: Visualizing all tokens in Dante's Divine Comedy



In [ ]:
# download the corpus from https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt
import requests

url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
response = requests.get(url)
with open("commedia.txt", "w") as f:
    f.write(response.text)

dataset = load_dataset("text", data_files={"train": "commedia.txt"})
corpus = " ".join(dataset["train"]["text"])

from collections import Counter

# Tokenize the corpus
input_ids = tokenizer(corpus, return_tensors="pt")["input_ids"].flatten()
unique_ids = sorted(set(input_ids.tolist()))
freqs = Counter(input_ids.tolist())

# Extract embeddings only for tokens in corpus
embedding_weights = model.model.embed_tokens(torch.tensor(unique_ids).unsqueeze(0).to(model.device)).squeeze(0)

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Reduce to 3D
reduce_dims = PCA(n_components=3)
# reduce_dims = TSNE(n_components=3)

embeddings_3d = reduce_dims.fit_transform(embedding_weights.detach().cpu().float())

# We convert token IDs back into human-readable text and collect their frequencies.
tokens = [tokenizer.decode([i]) for i in unique_ids]
sizes = [freqs[i] for i in unique_ids]



Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
import plotly.io as pio
import plotly.express as px

pio.renderers.default = 'colab'

# Plot with Plotly
fig = px.scatter_3d(
    x=embeddings_3d[:, 0],
    y=embeddings_3d[:, 1],
    z=embeddings_3d[:, 2],
    hover_name=tokens,
    color={k: v if v < 100 else 100 for k, v in freqs.items()}, # cutting the frequency to 100 because there are too many outliers
    #size={k: v if v < 100 else 100 for k, v in freqs.items()},
    title="Token Embeddings (Filtered by Corpus)",
)

fig.update_traces(marker=dict(size=2))
fig.update_coloraxes(showscale=False)

fig.update_layout(
    scene=dict(
        aspectmode="cube",
        xaxis=dict(backgroundcolor="white", gridcolor="lightgray"),
        yaxis=dict(backgroundcolor="white", gridcolor="lightgray"),
        zaxis=dict(backgroundcolor="white", gridcolor="lightgray"),
    ),
    paper_bgcolor="#ebebeb",  # around the plot
    font_color="white"      # text, labels, hover
)

fig.show()

## 🚀 Extra 2: (Optional) Visualize the Entire Vocabulary


In [ ]:

# full_ids = list(range(tokenizer.vocab_size))
# full_embedding_weights = model.embed_tokens(torch.tensor(full_ids).unsqueeze(0).to(model.device)).squeeze(0)
# full_embeddings_3d = PCA(n_components=3).fit_transform(full_embedding_weights.cpu().detach().float())
# full_tokens = [tokenizer.decode([i]) for i in full_ids]
#
# fig_full = px.scatter_3d(
#     x=full_embeddings_3d[:, 0],
#     y=full_embeddings_3d[:, 1],
#     z=full_embeddings_3d[:, 2],
#     hover_name=full_tokens,
#     title="Token Embeddings (Full Vocabulary)"
# )
# fig_full.update_traces(marker=dict(size=2))
# fig_full.show()